<a href="https://colab.research.google.com/github/brendonhuynhbp-hub/gt-markets/blob/main/notebooks/app/01_prepare_app_data_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
# === BLOCK 1: Prepare artefacts for Streamlit app ===
import os, shutil, glob
from pathlib import Path
import pandas as pd
import numpy as np

# --- Paths ---
from google.colab import drive
drive.mount("/content/drive")

DATA_DIR = Path("/content/drive/MyDrive/gt-markets/data/processed")
IN_CSV   = DATA_DIR / "merged_financial_trends_engineered_2025-09-07.csv"

DRIVE_ARTEFACTS = Path("/content/drive/MyDrive/gt-markets/AppDemo/artefacts")
REPO_ARTEFACTS  = Path("/content/gt-markets/AppDemo/artefacts")

for p in [DRIVE_ARTEFACTS, REPO_ARTEFACTS]:
    p.mkdir(parents=True, exist_ok=True)

print("Input:", IN_CSV)
print("Drive artefacts:", DRIVE_ARTEFACTS)
print("Repo artefacts:", REPO_ARTEFACTS)

# --- Load merged data ---
df = pd.read_csv(IN_CSV, parse_dates=["Date"]).set_index("Date").sort_index()
print("Merged shape:", df.shape)
print("Sample cols:", list(df.columns[:12]))

# --- Assets and strategies ---
assets = {
    "GOLD": "GC=F Close",
    "BTC": "BTC-USD Close",
    "OIL": "CL=F Close",
    "USDCNY": "USDCNY=X Close",
}
strategies = {
    "sma": lambda x: x.rolling(20).mean(),    # simple moving avg
    "ema": lambda x: x.ewm(span=20).mean(),   # exponential moving avg
}
freqs = {"D": "1D", "W": "1W"}

# --- Helpers ---
def calc_baseline(close: pd.Series, strategy: str) -> dict:
    """Baseline backtest for one strategy on given price series"""
    ret = close.pct_change().fillna(0)
    if strategy == "sma":
        signal = (close > close.rolling(20).mean()).astype(int)
    elif strategy == "ema":
        signal = (close > close.ewm(span=20).mean()).astype(int)
    else:
        raise ValueError(strategy)
    strat_ret = ret * signal.shift(1).fillna(0)
    return {
        "strategy": strategy,
        "type": "baseline",
        "cagr": (1+strat_ret).prod()**(252/len(strat_ret)) - 1,
        "sharpe": strat_ret.mean() / strat_ret.std(ddof=0) * np.sqrt(252) if strat_ret.std() else 0,
        "trades": int((signal.diff().abs() > 0).sum())
    }

def calc_keyword(sig: pd.Series, close: pd.Series, strategy: str) -> dict:
    """Keyword-based metrics: use ML prob_up signal + trading strategy"""
    sig = sig.fillna(0).clip(0, 1)  # prob_up between 0–1
    ret = close.pct_change().fillna(0)
    if strategy == "sma":
        base = close.rolling(20).mean()
        signal = (sig > 0.5) & (close > base)
    elif strategy == "ema":
        base = close.ewm(span=20).mean()
        signal = (sig > 0.5) & (close > base)
    else:
        raise ValueError(strategy)
    signal = signal.astype(int)
    strat_ret = ret * signal.shift(1).fillna(0)
    return {
        "strategy": strategy,
        "type": "keyword",
        "cagr": (1+strat_ret).prod()**(252/len(strat_ret)) - 1,
        "sharpe": strat_ret.mean() / strat_ret.std(ddof=0) * np.sqrt(252) if strat_ret.std() else 0,
        "trades": int((signal.diff().abs() > 0).sum())
    }

# --- Export artefacts ---
summary_rows = []
for asset, col in assets.items():
    closeD = df[col].resample("1D").last().dropna()
    closeW = df[col].resample("1W").last().dropna()

    asset_dir = REPO_ARTEFACTS / asset
    if asset_dir.exists():
        shutil.rmtree(asset_dir)
    asset_dir.mkdir(parents=True)

    for freq, rule in freqs.items():
        close = closeD if freq == "D" else closeW

        # --- Baseline ---
        rows = []
        for strat in strategies:
            m = calc_baseline(close, strat)
            rows.append(m)
            summary_rows.append({"asset": asset, "freq": freq, **m})
        dfb = pd.DataFrame(rows)
        f_out = asset_dir / f"metrics_baseline_{freq}.csv"
        dfb.to_csv(f_out, index=False)
        print(f"[ok] {asset} {freq}: baseline rows={len(dfb)}")

        # --- Keyword (loop ML/DL signal files) ---
        kw_rows = []
        for sig_file in glob.glob(str(asset_dir.parent / asset / f"signals_KW_*_{freq}.csv")):
            sig = pd.read_csv(sig_file, parse_dates=["date"]).set_index("date")["signal"]
            for strat in strategies:
                m = calc_keyword(sig, close, strat)
                kw_rows.append({"model": Path(sig_file).stem, **m})
                summary_rows.append({"asset": asset, "freq": freq, "model": Path(sig_file).stem, **m})
        if kw_rows:
            dfk = pd.DataFrame(kw_rows)
            f_out = asset_dir / f"metrics_keywords_{freq}.csv"
            dfk.to_csv(f_out, index=False)
            print(f"[ok] {asset} {freq}: keywords rows={len(dfk)}")

# --- Summaries at root ---
pd.DataFrame(summary_rows).to_csv(REPO_ARTEFACTS/"metrics_summary_D.csv", index=False)
pd.DataFrame(summary_rows).to_csv(REPO_ARTEFACTS/"metrics_summary_W.csv", index=False)

# Copy artefacts to Drive for backup
if DRIVE_ARTEFACTS.exists():
    shutil.rmtree(DRIVE_ARTEFACTS)
shutil.copytree(REPO_ARTEFACTS, DRIVE_ARTEFACTS)

# --- Diagnostics ---
for f in sorted(glob.glob(str(REPO_ARTEFACTS/"*/*.csv"))):
    sz = os.stat(f).st_size
    try:
        rows = len(pd.read_csv(f))
    except Exception:
        rows = -1
    print(f"{f}  size={sz}  rows={rows}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Input: /content/drive/MyDrive/gt-markets/data/processed/merged_financial_trends_engineered_2025-09-07.csv
Drive artefacts: /content/drive/MyDrive/gt-markets/AppDemo/artefacts
Repo artefacts: /content/gt-markets/AppDemo/artefacts
Merged shape: (2609, 254)
Sample cols: ['BTC-USD Close', 'CL=F Close', 'DXY Close', 'GC=F Close', 'USDCNY=X Close', 'BTC-USD Open', 'CL=F Open', 'DXY Open', 'GC=F Open', 'USDCNY=X Open', 'BTC-USD High', 'CL=F High']
[ok] GOLD D: baseline rows=2
[ok] GOLD W: baseline rows=2
[ok] BTC D: baseline rows=2
[ok] BTC W: baseline rows=2
[ok] OIL D: baseline rows=2
[ok] OIL W: baseline rows=2
[ok] USDCNY D: baseline rows=2
[ok] USDCNY W: baseline rows=2
/content/gt-markets/AppDemo/artefacts/BTC/metrics_baseline_D.csv  size=143  rows=2
/content/gt-markets/AppDemo/artefacts/BTC/metrics_baseline_W.csv  size=139  rows=2
/content/gt-markets/AppDemo/

In [23]:
# --- BLOCK 2: Sync artefacts to GitHub (always prompt for PAT) ---
import os, sys, shutil, subprocess, time, getpass, urllib.parse
from pathlib import Path

# ==== Config ====
DRIVE_ARTEFACTS = Path("/content/drive/MyDrive/gt-markets/AppDemo/artefacts")
REPO_ROOT       = Path("/content/gt-markets")
REPO_ARTEFACTS  = REPO_ROOT / "AppDemo/artefacts"
GITHUB_OWNER    = "brendonhuynhbp-hub"
GITHUB_REPO     = "gt-markets"
BRANCH          = "main"

# ==== Helpers ====
def log(msg): print(msg, flush=True)

def sh(cmd, cwd=None, timeout=None, check=False, hide=False):
    """Run a shell command and stream output. Returns returncode."""
    if not hide:
        print("$ " + " ".join(cmd))
    proc = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True, timeout=timeout)
    if not hide:
        if proc.stdout: print(proc.stdout, end="")
        if proc.stderr: print(proc.stderr, end="")
    if check and proc.returncode != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd, proc.stdout, proc.stderr)
    return proc.returncode, proc.stdout, proc.stderr

def safe_rmtree(p: Path):
    if p.exists():
        shutil.rmtree(p)

def copy_tree(src: Path, dst: Path):
    """Copy src tree onto dst (create/overwrite, keep structure)."""
    if not src.exists():
        raise FileNotFoundError(f"Source not found: {src}")
    dst.mkdir(parents=True, exist_ok=True)
    copied = 0
    for root, dirs, files in os.walk(src):
        r = Path(root)
        rel = r.relative_to(src)
        (dst / rel).mkdir(parents=True, exist_ok=True)
        for f in files:
            s = r / f
            d = dst / rel / f
            shutil.copy2(s, d)
            copied += 1
            print(f"[copied] {s} -> {d}")
    return copied

# ==== 0) Ask for PAT (ALWAYS) ====
print("This step needs a GitHub Personal Access Token (scope: repo).")
print("Tip: For GH orgs with SSO, enable the token for that org.")
PAT = getpass.getpass("GitHub Personal Access Token (PAT): ")
if not PAT:
    raise SystemExit("No PAT provided. Aborting.")

# Build a PAT-embedded remote URL (we’ll remove it afterward)
safe_pat = urllib.parse.quote(PAT, safe="")
REMOTE_PAT = f"https://{safe_pat}@github.com/{GITHUB_OWNER}/{GITHUB_REPO}.git"
REMOTE_PLAIN = f"https://github.com/{GITHUB_OWNER}/{GITHUB_REPO}.git"

# ==== 1) Ensure repo exists & is clean ====
if REPO_ROOT.exists() and not (REPO_ROOT / ".git").exists():
    log("⚠️ Found existing repo path without .git/ — removing so we can clone clean.")
    safe_rmtree(REPO_ROOT)

if not REPO_ROOT.exists():
    log("Repo not found at /content/gt-markets. Attempting clone…")
    REPO_ROOT.mkdir(parents=True, exist_ok=True)
    rc, out, err = sh(["git", "clone", REMOTE_PAT, str(REPO_ROOT)])
    if rc != 0:
        raise RuntimeError("Clone failed. Check PAT scopes and SSO access.")
else:
    log("Repo already present at /content/gt-markets.")

# ==== 2) Configure git identity & remote ====
print(f"Repo root: {REPO_ROOT}")
sh(["git", "config", "user.name", GITHUB_OWNER], cwd=REPO_ROOT)
sh(["git", "config", "user.email", "you@example.com"], cwd=REPO_ROOT)

# Set remote to PAT URL (temporary)
sh(["git", "remote", "set-url", "origin", REMOTE_PAT], cwd=REPO_ROOT)

# Checkout correct branch
sh(["git", "fetch", "origin", "--prune"], cwd=REPO_ROOT)
# Create branch locally if missing
rc, out, _ = sh(["git", "rev-parse", "--verify", BRANCH], cwd=REPO_ROOT)
if rc != 0:
    sh(["git", "checkout", "-b", BRANCH, f"origin/{BRANCH}"], cwd=REPO_ROOT)
else:
    sh(["git", "checkout", BRANCH], cwd=REPO_ROOT)

# Pull (rebase) or auto-heal
log("\n== Fetching and rebasing on latest remote ==")
rc, _, _ = sh(["git", "pull", "--rebase", "origin", BRANCH], cwd=REPO_ROOT)
if rc != 0:
    print("Pull --rebase failed.\nApplying automatic fix: stash + hard reset to origin/<branch> …")
    sh(["git", "stash", "push", "-u", "-m", "colab-autostash"], cwd=REPO_ROOT)
    sh(["git", "checkout", BRANCH], cwd=REPO_ROOT)
    sh(["git", "reset", "--hard", f"origin/{BRANCH}"], cwd=REPO_ROOT)

# ==== 3) Sync artefacts from Drive -> repo ====
log("\n== Syncing from Drive -> repo ==")
copied_count = copy_tree(DRIVE_ARTEFACTS, REPO_ARTEFACTS)
print(f"== Sync complete (files copied: {copied_count}) ==\n")

# ==== 4) Stage, commit, push ====
log("== Staging: AppDemo/artefacts ==")
rc, out, _ = sh(["git", "add", "-A", "AppDemo/artefacts"], cwd=REPO_ROOT)
# Check if there’s anything to commit
rc, status, _ = sh(["git", "status", "--porcelain", "AppDemo/artefacts"], cwd=REPO_ROOT, hide=True)
if status.strip() == "":
    print("No changes to commit in AppDemo/artefacts.")
else:
    msg = "Update artefacts (Colab sync)"
    sh(["git", "commit", "-m", msg], cwd=REPO_ROOT, check=True)
    print("\n== Pushing to remote ==")
    rc, _, _ = sh(["git", "push", "origin", BRANCH], cwd=REPO_ROOT)
    if rc != 0:
        raise RuntimeError("git push failed. Check PAT scopes or branch protection.")
    print(f"✅ Pushed AppDemo/artefacts to https://github.com/{GITHUB_OWNER}/{GITHUB_REPO} on '{BRANCH}'")

# ==== 5) Security: restore plain remote (remove PAT from local config) ====
sh(["git", "remote", "set-url", "origin", REMOTE_PLAIN], cwd=REPO_ROOT)
print("\n🔒 Remote URL cleaned (PAT removed from local git config). Done.")


This step needs a GitHub Personal Access Token (scope: repo).
Tip: For GH orgs with SSO, enable the token for that org.
GitHub Personal Access Token (PAT): ··········
Repo already present at /content/gt-markets.
Repo root: /content/gt-markets
$ git config user.name brendonhuynhbp-hub
$ git config user.email you@example.com
$ git remote set-url origin https://%23%20---%20BLOCK%202%3A%20Sync%20artefacts%20to%20GitHub%20%28always%20prompt%20for%20PAT%29%20---%20import%20os%2C%20sys%2C%20shutil%2C%20subprocess%2C%20time%2C%20getpass%2C%20urllib.parse%20from%20pathlib%20import%20Path%20%20%23%20%3D%3D%3D%3D%20Config%20%3D%3D%3D%3D%20DRIVE_ARTEFACTS%20%3D%20Path%28%22%2Fcontent%2Fdrive%2FMyDrive%2Fgt-markets%2FAppDemo%2Fartefacts%22%29%20REPO_ROOT%20%20%20%20%20%20%20%3D%20Path%28%22%2Fcontent%2Fgt-markets%22%29%20REPO_ARTEFACTS%20%20%3D%20REPO_ROOT%20%2F%20%22AppDemo%2Fartefacts%22%20GITHUB_OWNER%20%20%20%20%3D%20%22brendonhuynhbp-hub%22%20GITHUB_REPO%20%20%20%20%20%3D%20%22gt-markets%22%20